# Catboost
O principal do catboost é que ele processa dados catégoricos(nome,descrição,...) sem a necessidade de "modelamento" por meio de **CTR (Categorical Target Statistics)**.
o CTR é a transformação da dado catégorico em valor númerico e relaciona com a váriavél alvo

- Árvores simétricas
- Suporte a dados NaN
- SHAP: Mostra como cada atributo influenciou na decisão tomada

In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

## Instalação da biblioteca e carregamento da OneBigTable

In [2]:
!pip3 install catboost
!pip3 install scikit-learn
!pip3 install pyarrow
!pip3 install optuna
!pip3 install pandas


In [3]:
import catboost as cat
import sklearn as sk
import pandas as pd
import os

import numpy as np
import optuna

from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
)

In [4]:
import os
pathKaggle = "../input/datasets/luisfilipelima/bancotransacao/"
path = "../../data/processed/"
pathDrive = "/content/drive/MyDrive/TreinosIAS/catboost/dados/"
pathDrive=path
path = pathKaggle
path

'../input/datasets/luisfilipelima/bancotransacao/'

In [5]:
#utilizar apenas no kaggle
import torch

print("GPU disponível:", torch.cuda.is_available())
print("Quantidade de GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


GPU disponível: True
Quantidade de GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [6]:
#Caso não esteja funcinando tente reiniciar o kernel
obt = pd.read_parquet(path+"obt_with_terminal_history.parquet")
obtTrain = pd.read_parquet(path+"obt_train.parquet")
obtTest = pd.read_parquet(path+"obt_validation.parquet")
obtTrain

,transaction_id,tx_datetime,tx_amount,card_id,terminal_id,terminal_category,hour,day_week,terminal_age_days,is_fraud,...,terminal_tx_count_24h,terminal_distinct_cards_24h,seconds_since_last_terminal_tx,card_history_count_30d,card_amount_mean_30d,card_amount_std_30d,amount_z_score_30d,card_distinct_terminals,terminal_fraud_rate,card_fraud_rate
0,1156634,2018-02-15 13:34:00,12.400000,3520842735991929,847,SmartTalk Communications,13,4,173.0,False,...,24,23,1396.0,165,12.378606,8.682511,0.002464,155,0.001805,0.003953
1,1157370,2018-02-15 13:57:03,52.340000,370934368869147,847,SmartTalk Communications,13,4,173.0,False,...,25,24,1383.0,123,105.470894,91.224616,-0.582418,134,0.001803,0.000000
2,1157375,2018-02-15 13:57:17,78.169998,6011689217447071,847,SmartTalk Communications,13,4,173.0,False,...,26,25,14.0,122,139.657131,132.426997,-0.464310,168,0.001802,0.000000
3,1159536,2018-02-15 15:09:34,23.850000,2662393784087193,847,SmartTalk Communications,15,4,173.0,False,...,24,23,4337.0,142,91.744367,119.426299,-0.568504,149,0.001800,0.000000
4,1160251,2018-02-15 15:35:49,29.459999,3515828436568584,847,SmartTalk Communications,15,4,173.0,False,...,24,23,1575.0,102,43.886078,27.739705,-0.520052,113,0.001799,0.016575
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3428899,1037845,2018-02-11 08:47:36,174.339996,30245744153961,1327,Elite Hair Salon,8,0,15.0,False,...,25,25,1403.0,118,162.109661,159.539498,0.076660,95,0.000000,0.000000
3428900,1039951,2018-02-11 10:01:18,64.620003,213135672071906,1327,Elite Hair Salon,10,0,15.0,False,...,25,25,4422.0,34,36.808235,50.233755,0.553647,41,0.000000,0.000000
3428901,1042149,2018-02-11 11:11:48,176.210007,213144998564727,1327,Elite Hair Salon,11,0,15.0,False,...,25,25,4230.0,188,78.177340,52.274260,1.875353,193,0.000000,0.004016
3428902,1042756,2018-02-11 11:30:50,159.520004,4951962452405584,1327,Elite Hair Salon,11,0,15.0,False,...,26,26,1142.0,194,81.501082,94.760116,0.823331,172,0.000000,0.003937


In [7]:
cat_features = ["terminal_category"]

features = [
    "tx_amount",
    "hour",
    "day_week",
    "terminal_age_days",
    "terminal_fraud_rate",
    "card_fraud_rate",
    "card_distinct_terminals",
    "terminal_new_for_card",
    "card_tx_count_1h",
    "card_tx_count_24h",
    "card_distinct_terminals_24h",
    "seconds_since_last_card_tx",
    "terminal_tx_count_1h",
    "terminal_tx_count_24h",
    "terminal_distinct_cards_24h",
    "seconds_since_last_terminal_tx",
    "card_history_count_30d",
    "card_amount_mean_30d",
    "card_amount_std_30d",
    "amount_z_score_30d",
    "terminal_category",
]
target = "is_fraud"
def prepararParaCatboost(df):
    df["datetime"] = pd.to_datetime(df["tx_datetime"], errors="raise")
    df[target] = df[target].astype(int)

    # CatBoost exige categorias sem valores nulos.
    df["terminal_category"] = (
        df["terminal_category"].fillna("SEM_CATEGORIA").astype(str)
    )

    numeric_features = [c for c in features if c not in cat_features]
    for col in numeric_features:
        df[col] = pd.to_numeric(df[col], errors="raise").astype(float)

    df = df.sort_values("tx_datetime").reset_index(drop=True)
    return df
    
obt = prepararParaCatboost(obt)
obtTrain = prepararParaCatboost(obtTrain)
obtTest = prepararParaCatboost(obtTest)


## Hiperparâmetros
Como o catboost usa técnica de boosting, seus hiperparametros ajustam tanto as árvores quanto o nível das correções feitas

| Hiperparâmetro | O que controla | Efeito prático |
|---|---|---|
| `iterations` | Número máximo de árvores | Mais árvores permitem aprender mais, mas aumentam o tempo e podem causar sobreajuste. |
| `learning_rate` | Peso da contribuição de cada nova árvore | Valores menores tornam o aprendizado gradual e normalmente exigem mais árvores. |
| `depth` | Profundidade das árvores | Valores maiores capturam relações complexas, mas aumentam o custo e o risco de sobreajuste. |
| `l2_leaf_reg` | Penalização dos valores das folhas | Valores maiores suavizam as previsões e restringem o ajuste aos dados de treino. |
| `random_strength` | Aleatoriedade na escolha das divisões das árvores | Pode reduzir o sobreajuste; em excesso, prejudica o aprendizado. |
| `bagging_temperature` | Variação dos pesos das observações no bootstrap bayesiano | `0` dá pesos iguais; valores maiores aumentam a aleatoriedade. Exige `bootstrap_type="Bayesian"`. |
| `border_count` | Quantidade de limites usados para discretizar variáveis numéricas | Mais limites permitem distinções mais finas, com possível aumento de custo. |

### Optuna
É uma biblioteca de automátização para encontrar os melhores hiperparâmetros definindo a qualidade de resposta do modelo, quais hiperparemetros variam e tempo gasto
### TPE
O TPE gera os valores dos hipermarametos usados para o teste, porém esse hiperparâmetros são escolhidos/gerados a partir dos resultados dos testes anteriores


In [8]:
def calcular_lucro(TransacoesAceitas):
    if TransacoesAceitas.empty:
        print("Nenhuma transação aceita; lucro = 0.")
        return 0.0
    legitimas = TransacoesAceitas["is_fraud"].eq(0)
    fraudulentas = TransacoesAceitas["is_fraud"].eq(1)

    receita = (
        0.03 * TransacoesAceitas.loc[legitimas, "tx_amount"].sum()
    )

    prejuizo = (
        TransacoesAceitas.loc[fraudulentas, "tx_amount"].sum()
    )
    print("CALCULANDO MELHOR LUCRO...")
    print(f"VALOR:{float(receita - prejuizo)}")
    # Recusas têm resultado financeiro direto igual a zero.
    return float(receita - prejuizo)

def classificarTransacoes(x,y,prob,limiar):
    transacoes = x.copy()
    transacoes["is_fraud"] = np.asarray(y, dtype=int)
    transacoes["probabilidade_fraude"] = prob

    aceita = transacoes["probabilidade_fraude"] < limiar

    TransacoesAceitas = transacoes.loc[aceita].copy()
    TransacoesRecusadas = transacoes.loc[~aceita].copy() # pega todas menos as acceitas

    return TransacoesAceitas, TransacoesRecusadas

def encontrar_melhor_limiar(modelo, X_val, y_val):
    probabilidades = modelo.predict_proba(X_val)[:, 1]
    valores = X_val["tx_amount"].to_numpy()
    y = np.asarray(y_val, dtype=int)

    # Ordena da menor para a maior chance de fraude. Assim cada posição
    # representa o lucro acumulado ao aceitar as transações até aquele ponto.
    ordem = np.argsort(probabilidades, kind="stable")
    probabilidades_ordenadas = probabilidades[ordem]
    valores_ordenados = valores[ordem]
    y_ordenado = y[ordem]

    lucro_por_transacao = np.where(
        y_ordenado == 0,
        0.03 * valores_ordenados,
        -valores_ordenados,
    )
    lucro_acumulado = np.cumsum(lucro_por_transacao)

    # Probabilidades iguais precisam ser aceitas ou recusadas juntas.
    fim_de_grupo = np.concatenate([
        probabilidades_ordenadas[1:] != probabilidades_ordenadas[:-1],
        [True],
    ])
    indices = np.flatnonzero(fim_de_grupo)
    lucros = lucro_acumulado[indices]

    # Também compara com a opção de recusar todas as transações.
    lucros = np.concatenate([[0.0], lucros])
    melhor_posicao = int(np.argmax(lucros))
    melhor_lucro = float(lucros[melhor_posicao])

    if melhor_posicao == 0:
        melhor_limiar = np.nextafter(
            probabilidades_ordenadas[0], -np.inf
        )
    else:
        indice = indices[melhor_posicao - 1]
        melhor_limiar = np.nextafter(
            probabilidades_ordenadas[indice], np.inf
        )

    return float(melhor_limiar), melhor_lucro


### Busca bayaseiana
ao invés de treinar utilizando o conjunto de testes completos, é melhor colocar uma pequena parte para treinar inicialmente, assim os hiperparametros se ajustam <br>
A busca  bayesiana vai procurar estimiar melhores valores de hiperparametros ao decorrer o teste

In [9]:
# Janeiro a março formam a janela cronológica da busca walk-forward.
# Abril escolhe o limiar e maio testa o resultado final.
obtLimiar = obtTrain.loc[lambda x: x["datetime"].dt.month == 4].copy()

# A busca e o treino final usam somente dados anteriores a abril.
obtTrain = obtTrain.loc[lambda x: x["datetime"].dt.month <= 3].copy()


In [10]:
# Usa as linhas mais recentes sem mudar artificialmente a proporção de fraudes.
N_SPLITS = 10
Buscatrain = obtTrain.tail(300_000).copy()

print("Linhas usadas na busca:", len(Buscatrain))
print("Taxa de fraude na busca:", Buscatrain["is_fraud"].mean())
print("Taxa de fraude no treino original:", obtTrain["is_fraud"].mean())


Linhas usadas na busca: 300000
Taxa de fraude na busca: 0.011633333333333334
Taxa de fraude no treino original: 0.008757633339829794


In [11]:
test = obtTest

X_train, y_train = obtTrain[features], obtTrain[target]
X_limiar, y_limiar = obtLimiar[features], obtLimiar[target]
X_test, y_test = test[features], test[target]

# Mesma janela de busca do XGBoost: 300 mil registros mais recentes.
X_busca = Buscatrain[features].copy().reset_index(drop=True)
y_busca = Buscatrain[target].copy().reset_index(drop=True)



In [12]:
SEED = 42
N_INIT_TRAILS = 20
N_TRIALS = 100



fixed_params = {
    "loss_function": "Logloss",
    "eval_metric": "PRAUC",
    "bootstrap_type": "Bayesian",
    "random_seed": SEED,
    "thread_count": 4,
    "verbose": False,
    "allow_writing_files": False,
}

def objective(trial):
    params = {
        "iterations": 300,
        "depth": trial.suggest_int("depth", 4, 6),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.20, log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 0.1, 100.0, log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.0, 5.0
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0.0, 5.0
        ),
    }

    class_weight_strategy = trial.suggest_categorical(
        "class_weight_strategy", ["none", "SqrtBalanced", "Balanced"]
    )
    if class_weight_strategy != "none":
        params["auto_class_weights"] = class_weight_strategy

    lucros_folds = []
    prauc_folds = []
    iteracoes_folds = []
    splitter = TimeSeriesSplit(n_splits=N_SPLITS)

    # Cada validação acontece depois do seu treino; as janelas de treino crescem.
    for train_idx, val_idx in splitter.split(X_busca):
        X_fit = X_busca.iloc[train_idx]
        y_fit = y_busca.iloc[train_idx]
        X_val = X_busca.iloc[val_idx]
        y_val = y_busca.iloc[val_idx]

        model = CatBoostClassifier(**fixed_params, **params, task_type="GPU", devices="0")
        model.fit(
            X_fit,
            y_fit,
            cat_features=cat_features,
            eval_set=(X_val, y_val),
            early_stopping_rounds=40,
            use_best_model=True,
        )

        probabilidades = model.predict_proba(X_val)[:, 1]
        _, lucro_fold = encontrar_melhor_limiar(model, X_val, y_val)
        lucros_folds.append(lucro_fold)
        prauc_folds.append(average_precision_score(y_val, probabilidades))
        iteracoes_folds.append(model.tree_count_)

    trial.set_user_attr("lucros_por_fold", lucros_folds)
    trial.set_user_attr("prauc_por_fold", prauc_folds)
    trial.set_user_attr("iteracoes_por_fold", iteracoes_folds)
    trial.set_user_attr("lucro_medio", float(np.mean(lucros_folds)))
    trial.set_user_attr("prauc_media", float(np.mean(prauc_folds)))

    return float(np.mean(lucros_folds))

#utilizacao do TPE
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=N_INIT_TRAILS,
    ),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\nMelhor lucro de validação:", study.best_value)
print("Melhores hiperparâmetros:", study.best_params)
print("Lucros por fold:", study.best_trial.user_attrs["lucros_por_fold"])
print("PRAUC médio dos folds:", study.best_trial.user_attrs["prauc_media"])
print("Iterações por fold:", study.best_trial.user_attrs["iteracoes_por_fold"])


[I 2026-11-12 13:18:51,447] A new study created in memory with name: no-name-3c1e7a2d-0b3b-46df-bb82-e6224cf5a479


  0%|          | 0/100 [00:00<?, ?it/s]

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 

[I 2026-11-12 13:19:08,522] Trial 0 finished with value: 50786.85709937929 and parameters: {'depth': 5, 'learning_rate': 0.17254716573280354, 'l2_leaf_reg': 15.702970884055382, 'random_strength': 2.993292420985183, 'bagging_temperature': 0.7800932022121826, 'class_weight_strategy': 'Balanced'}. Best is trial 0 with value: 50786.85709937929.


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 

[I 2026-11-12 13:19:31,850] Trial 1 finished with value: 50898.03041927557 and parameters: {'depth': 5, 'learning_rate': 0.08341106432362087, 'l2_leaf_reg': 0.115279871282324, 'random_strength': 4.8495492608099715, 'bagging_temperature': 4.162213204002109, 'class_weight_strategy': 'none'}. Best is trial 1 with value: 50898.03041927557.


... (192 outputs left out)

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 

[I 2026-11-12 13:53:00,083] Trial 98 finished with value: 50911.810039490636 and parameters: {'depth': 4, 'learning_rate': 0.06669411164872244, 'l2_leaf_reg': 2.1899613414038672, 'random_strength': 0.9777861688364644, 'bagging_temperature': 0.1430452447425356, 'class_weight_strategy': 'none'}. Best is trial 93 with value: 50914.81975092595.


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Default metric period is 

[I 2026-11-12 13:53:20,662] Trial 99 finished with value: 50893.48639934506 and parameters: {'depth': 4, 'learning_rate': 0.05962613619990066, 'l2_leaf_reg': 0.9549861260291943, 'random_strength': 1.0111687919979113, 'bagging_temperature': 0.6864711702614468, 'class_weight_strategy': 'none'}. Best is trial 93 with value: 50914.81975092595.

Melhor lucro de validação: 50914.81975092595
Melhores hiperparâmetros: {'depth': 4, 'learning_rate': 0.05782969162827572, 'l2_leaf_reg': 1.2789335317947779, 'random_strength': 1.0354358669169343, 'bagging_temperature': 0.6186864295597393, 'class_weight_strategy': 'none'}
Lucros por fold: [51532.032384779406, 50435.39380747497, 50854.529400534026, 50117.64569189842, 48776.54971088253, 53263.204870514026, 50227.48187166784, 50712.01840979584, 50456.027132899864, 52773.31422881251]
PRAUC médio dos folds: 0.494084470115682
Iterações por fold: [45, 117, 26, 26, 84, 34, 56, 60, 141, 69]


In [20]:
print("teste")

teste


In [21]:
melhores_parametros = study.best_params.copy()

estrategia_pesos = melhores_parametros.pop(
    "class_weight_strategy"
)

if estrategia_pesos != "none":
    melhores_parametros["auto_class_weights"] = estrategia_pesos

modelo_final = CatBoostClassifier(
    **fixed_params,
    **melhores_parametros
   # iterations=study.best_trial.user_attrs["n_trees"]
)

modelo_final.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    verbose=False
)

# O limiar é escolhido depois do treino final, usando apenas abril.
limiar_final, lucro_limiar = encontrar_melhor_limiar(
    modelo_final,
    X_limiar,
    y_limiar,
)
modelo_final.set_probability_threshold(limiar_final)

print("Melhor limiar:", limiar_final)
print("Lucro em abril:", lucro_limiar)


Melhor limiar: 0.03710451615357564
Lucro em abril: 1777077.0869721796


In [22]:
probabilidades_teste = modelo_final.predict_proba(X_test)[:, 1]
predicoes_teste = (probabilidades_teste >= limiar_final).astype(int)

resultado_teste = X_test.copy()

resultado_teste["probabilidade_fraude"] = probabilidades_teste
resultado_teste["fraude_prevista"] = predicoes_teste
resultado_teste["fraude_real"] = y_test.to_numpy()

TransacoesAceitas, TransacoesRecusadas = classificarTransacoes(
    X_test,
    y_test,
    probabilidades_teste,
    limiar_final,
)

lucro_teste = calcular_lucro(TransacoesAceitas)
matriz = confusion_matrix(y_test, predicoes_teste)
precisao = precision_score(y_test, predicoes_teste, zero_division=0)
recall = recall_score(y_test, predicoes_teste, zero_division=0)
prauc_teste = average_precision_score(y_test, probabilidades_teste)

print("Matriz de confusão:")
print(matriz)
print("Lucro em maio:", lucro_teste)
print("Taxa de transações recusadas:", len(TransacoesRecusadas) / len(X_test))

res = resultado_teste.loc[
    (resultado_teste["fraude_real"] == 1)
    & (resultado_teste["fraude_prevista"] == 0)
]


res


CALCULANDO MELHOR LUCRO...
VALOR:1807350.3314370676
Matriz de confusão:
[[1038968    2850]
 [   7215    6019]]
Lucro em maio: 1807350.3314370676
Taxa de transações recusadas: 0.008406220736039551


,tx_amount,hour,day_week,terminal_age_days,terminal_fraud_rate,card_fraud_rate,card_distinct_terminals,terminal_new_for_card,card_tx_count_1h,card_tx_count_24h,...,terminal_distinct_cards_24h,seconds_since_last_terminal_tx,card_history_count_30d,card_amount_mean_30d,card_amount_std_30d,amount_z_score_30d,terminal_category,probabilidade_fraude,fraude_prevista,fraude_real
657,71.800003,1.0,2.0,353.0,0.005797,0.005900,192.0,0.0,0.0,4.0,...,11.0,14668.0,72.0,66.489444,66.052509,0.080399,Language Hub,0.004360,0,1
787,17.860001,2.0,2.0,75.0,0.005414,0.008671,378.0,0.0,0.0,8.0,...,18.0,20193.0,194.0,162.743196,179.568753,-0.806840,Taxi Carlos,0.006607,0,1
827,13.410000,2.0,2.0,231.0,0.003190,0.000000,183.0,1.0,0.0,0.0,...,13.0,24290.0,59.0,88.521186,96.395608,-0.779197,Prime Supermarket,0.002605,0,1
1056,54.029999,2.0,2.0,214.0,0.008434,0.012295,184.0,1.0,0.0,2.0,...,9.0,34407.0,54.0,68.011852,58.380180,-0.239497,Green Basket,0.004838,0,1
1274,266.410004,3.0,2.0,96.0,0.009079,0.002538,196.0,0.0,0.0,4.0,...,25.0,10494.0,96.0,75.360417,98.269782,1.944134,Smart Home Store,0.004693,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1054332,25.670000,22.0,4.0,65.0,0.006917,0.005995,452.0,0.0,0.0,11.0,...,18.0,5617.0,158.0,29.048164,18.911097,-0.178634,Creative Art Supplies,0.006518,0,1
1054784,323.220001,23.0,4.0,272.0,0.006423,0.002762,430.0,1.0,0.0,1.0,...,10.0,4660.0,146.0,55.021233,54.478007,4.923065,Gadget Galaxy,0.004893,0,1
1054853,116.650002,23.0,4.0,535.0,0.004984,0.009050,353.0,0.0,1.0,5.0,...,27.0,7443.0,120.0,95.846083,57.457696,0.362074,Elegant Carpets,0.004782,0,1
1054933,15.330000,23.0,4.0,97.0,0.003621,0.006826,219.0,1.0,0.0,4.0,...,17.0,3931.0,134.0,14.057090,9.147718,0.139151,Home Repair Hub,0.004648,0,1


In [23]:
import json

relatorio_hiperparametros = {
    "hiperparametros": study.best_params,
    "iterations": int(np.median(study.best_trial.user_attrs["iteracoes_por_fold"])),
    "lucro_medio_walk_forward": study.best_value,
    "lucros_por_fold": study.best_trial.user_attrs["lucros_por_fold"],
    "prauc_media_walk_forward": study.best_trial.user_attrs["prauc_media"],
}
print(json.dumps(relatorio_hiperparametros, indent=2, ensure_ascii=False))


{
  "hiperparametros": {
    "depth": 4,
    "learning_rate": 0.05782969162827572,
    "l2_leaf_reg": 1.2789335317947779,
    "random_strength": 1.0354358669169343,
    "bagging_temperature": 0.6186864295597393,
    "class_weight_strategy": "none"
  },
  "iterations": 58,
  "lucro_medio_walk_forward": 50914.81975092595,
  "lucros_por_fold": [
    51532.032384779406,
    50435.39380747497,
    50854.529400534026,
    50117.64569189842,
    48776.54971088253,
    53263.204870514026,
    50227.48187166784,
    50712.01840979584,
    50456.027132899864,
    52773.31422881251
  ],
  "prauc_media_walk_forward": 0.494084470115682
}


In [26]:
# O limiar configurado também fica salvo dentro do modelo.
modelo_final.save_model("catboostModel")
